<a href="https://colab.research.google.com/github/SilvanaCamboim/Desenvolvimento2025/blob/main/Aula03_Geopandas_Github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 55.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 63.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.9/23.9 MB 106.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 119.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 110.4 MB/s eta 0:00:00


In [3]:
import geopandas as gpd

# Carrega o shapefile dos municípios
municipios = gpd.read_file("https://raw.githubusercontent.com/SilvanaCamboim/Desenvolvimento2025/main/PR_Municipios_2023.geojson")



In [4]:
# Filtra o município de interesse
mun_selecionado = municipios[municipios["NM_MUN"] == "Castro"]
mun_selecionado.head()



,CD_MUN,NM_MUN,CD_RGI,NM_RGI,CD_RGINT,NM_RGINT,CD_UF,NM_UF,CD_REGIAO,NM_REGIAO,CD_CONCURB,NM_CONCURB,AREA_KM2,geometry
69,4104907,Castro,410027,Ponta Grossa,4106,Ponta Grossa,41,Paraná,4,Sul,None,None,2531.503,"MULTIPOLYGON (((-49.98416 -24.92276, -49.98519..."


In [5]:
# Verifica os vizinhos (usando touches)
vizinhos = municipios[municipios.touches(mun_selecionado.geometry.iloc[0])]
vizinhos.head()

,CD_MUN,NM_MUN,CD_RGI,NM_RGI,CD_RGINT,NM_RGINT,CD_UF,NM_UF,CD_REGIAO,NM_REGIAO,CD_CONCURB,NM_CONCURB,AREA_KM2,geometry
58,4104204,Campo Largo,410001,Curitiba,4101,Curitiba,41,Paraná,4,Sul,4106902,Curitiba/PR,1243.551,"MULTIPOLYGON (((-49.38596 -25.465, -49.38502 -..."
66,4104659,Carambeí,410027,Ponta Grossa,4106,Ponta Grossa,41,Paraná,4,Sul,4119905,Ponta Grossa/PR,649.679,"MULTIPOLYGON (((-50.11263 -25.00375, -50.11358..."
72,4105201,Cerro Azul,410001,Curitiba,4101,Curitiba,41,Paraná,4,Sul,None,None,1341.189,"MULTIPOLYGON (((-49.27599 -25.0517, -49.27898 ..."
159,4111258,Itaperuçu,410001,Curitiba,4101,Curitiba,41,Paraná,4,Sul,4106902,Curitiba/PR,322.991,"MULTIPOLYGON (((-49.35287 -25.19769, -49.35254..."
268,4119400,Piraí do Sul,410027,Ponta Grossa,4106,Ponta Grossa,41,Paraná,4,Sul,None,None,1345.418,"MULTIPOLYGON (((-49.95995 -24.61057, -49.96295..."


In [6]:
# Entrada do nome do município
mun = input("Digite o nome de um município: ")

# Filtra o município
selecionado = municipios[municipios['NM_MUN'] == mun]

# Reprojetar para o sistema de coordenadas adequado (29192)
selecionado_proj = selecionado.to_crs(epsg=29192)

# Calcular área em km²
selecionado_proj['area_km2'] = selecionado_proj.geometry.area / 10**6

# Exibir resultado
selecionado_proj[['NM_MUN', 'area_km2']].head()

Digite o nome de um município:  Castro


,NM_MUN,area_km2
69,Castro,2530.350798


In [15]:
from ipywidgets import Dropdown, Output
from IPython.display import display
import geopandas as gpd

# Supondo que você já carregou o GeoDataFrame `municipios`

# 1. Criar Dropdown com nomes e códigos dos municípios
comboMun = Dropdown(
    options=[(row['NM_MUN'], row['CD_MUN']) for _, row in municipios.sort_values('NM_MUN').iterrows()],
    description='Município:'
)

# 2. Criar uma saída separada
output = Output()

# 3. Função de callback usando output
def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        with output:
            output.clear_output()
            codigo = comboMun.value
            selecionado = municipios[municipios['CD_MUN'] == codigo]
            selecionado_proj = selecionado.to_crs(epsg=29192)
            area_km2 = selecionado_proj.geometry.area.iloc[0] / 1e6
            print(f"Área em km² do município selecionado: {area_km2:.2f}")

# 4. Associar função ao Dropdown
comboMun.observe(on_change)

# 5. Exibir interface
display(comboMun, output)



Dropdown(description='Município:', options=(('Abatiá', '4100103'), ('Adrianópolis', '4100202'), ('Agudos do Su…

Output()

In [16]:
from ipywidgets import Dropdown, Output
from IPython.display import display
import geopandas as gpd
import folium

# 1. Criar Dropdown
comboMun = Dropdown(
    options=[(row['NM_MUN'], row['CD_MUN']) for _, row in municipios.sort_values('NM_MUN').iterrows()],
    description='Município:'
)

# 2. Criar uma saída para exibir o resultado
output = Output()

# 3. Callback ao mudar a seleção
def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        with output:
            output.clear_output()
            codigo = comboMun.value

            # Filtrar município
            selecionado = municipios[municipios['CD_MUN'] == codigo]

            # Calcular área reprojetando para metros (EPSG:29101)
            selecionado_proj = selecionado.to_crs(epsg=29101)
            area_km2 = selecionado_proj.geometry.area.iloc[0] / 1e6
            print(f"Área em km² do município selecionado: {area_km2:.2f}")

            # Reprojetar para WGS84 (lat/lon) para exibir no folium
            selecionado_wgs84 = selecionado.to_crs(epsg=4326)

            # Centro do município
            centro = selecionado_wgs84.geometry.centroid.iloc[0]

            # Criar o mapa
            m = folium.Map(location=[centro.y, centro.x], zoom_start=10)

            folium.GeoJson(
                selecionado_wgs84,
                name="Município",
                style_function=lambda x: {
                    "fillColor": "blue",
                    "color": "blue",
                    "weight": 2,
                    "fillOpacity": 0.4
                }
            ).add_to(m)

            display(m)

# 4. Conectar a função ao Dropdown
comboMun.observe(on_change)

# 5. Exibir a interface
display(comboMun, output)




Dropdown(description='Município:', options=(('Abatiá', '4100103'), ('Adrianópolis', '4100202'), ('Agudos do Su…

Output()

In [10]:
!pip install folium

Dropdown(description='Município:', options=(('Abatiá', '4100103'), ('Adrianópolis', '4100202'), ('Agudos do Su…

In [18]:
from ipywidgets import Dropdown, Output
from IPython.display import display
import geopandas as gpd
import folium

# 1. Criar Dropdown com nomes e códigos dos municípios
comboMun = Dropdown(
    options=[(row['NM_MUN'], row['CD_MUN']) for _, row in municipios.sort_values('NM_MUN').iterrows()],
    description='Município:'
)

# 2. Criar uma saída para o resultado
output = Output()

# 3. Função para atualizar o mapa
def atualizar_mapa(change):
    if change['type'] == 'change' and change['name'] == 'value':
        with output:
            output.clear_output()
            codigo = comboMun.value

            # Município selecionado
            selecionado = municipios[municipios['CD_MUN'] == codigo]

            # Vizinhos (tocando o polígono)
            vizinhos = municipios[
                (municipios['CD_MUN'] != codigo) &
                (municipios.touches(selecionado.geometry.iloc[0]))
            ]

            # Reprojetar para WGS84
            selecionado_wgs84 = selecionado.to_crs(epsg=4326)
            vizinhos_wgs84 = vizinhos.to_crs(epsg=4326)

            # Centro do mapa
            centro = selecionado_wgs84.geometry.centroid.iloc[0]
            m = folium.Map(location=[centro.y, centro.x], zoom_start=10)

            # Adicionar município selecionado (azul)
            folium.GeoJson(
                selecionado_wgs84,
                name="Selecionado",
                style_function=lambda x: {"fillColor": "blue", "color": "blue", "weight": 2, "fillOpacity": 0.4}
            ).add_to(m)

            # Adicionar vizinhos (vermelho)
            folium.GeoJson(
                vizinhos_wgs84,
                name="Vizinhos",
                style_function=lambda x: {"fillColor": "red", "color": "red", "weight": 1.5, "fillOpacity": 0.3}
            ).add_to(m)

            folium.LayerControl().add_to(m)
            display(m)

# 4. Associar a função ao Dropdown
comboMun.observe(atualizar_mapa)

# 5. Mostrar interface
display(comboMun, output)



Dropdown(description='Município:', options=(('Abatiá', '4100103'), ('Adrianópolis', '4100202'), ('Agudos do Su…

Output()